# Green SM Food evaluation — CPU only

Attach exactly two Kaggle datasets: (1) `green_sm_food_poc20_v1`, containing the immutable Food test labels/candidate universe, and (2) the completed Food run output containing `green_sm_food_poc20_v1_r1.zip`. This notebook does not train or load a Foundation checkpoint.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path('/kaggle/working/repo')
if REPO.exists():
    raise RuntimeError('Fresh Kaggle session required: /kaggle/working/repo already exists.')

subprocess.run([
    'git', 'clone', '--branch', 'feat/food-poc-accelerated',
    'https://github.com/huongiang171/green-sm-foundation-pretraining.git', str(REPO),
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO, check=True)
print(subprocess.check_output(['git', 'log', '-1', '--oneline'], cwd=REPO, text=True).strip())


In [ ]:
import zipfile
from pathlib import Path, PurePosixPath

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/food_evaluation')

def kaggle_dataset_roots() -> list[Path]:
    # Support standard Kaggle mounts and this account's nested datasets/<owner>/<slug> layout.
    candidates = [
        INPUT_ROOT,
        *INPUT_ROOT.iterdir(),
        *INPUT_ROOT.glob('*/*'),
        *INPUT_ROOT.glob("*/*/*"),
    ]
    return [path for path in candidates if path.is_dir()]

def find_exact_root(name: str, required_relative: str) -> Path:
    matches = [
        root / name
        for root in kaggle_dataset_roots()
        if (root / name / required_relative).is_file()
    ]
    assert len(matches) == 1, f'Expected one {name!r}, found {matches}'
    return matches[0]

def kaggle_input_files() -> list[Path]:
    candidates = [
        *INPUT_ROOT.glob('*'),
        *INPUT_ROOT.glob('*/*'),
        *INPUT_ROOT.glob('*/*/*'),
        *INPUT_ROOT.glob('*/*/*/*'),
    ]
    return [path for path in candidates if path.is_file()]

def find_completed_run_roots() -> list[Path]:
    required = ('run_manifest.json', 'predictions/test/predictions.test.parquet')
    return [root for root in kaggle_dataset_roots() if all((root / item).is_file() for item in required)]

POC_ROOT = find_exact_root('green_sm_food_poc20_v1', 'food_opportunities/manifest.json')
RUN_ROOTS = find_completed_run_roots()
RUN_ARCHIVES = [path for path in kaggle_input_files() if path.name == 'green_sm_food_poc20_v1_r1.zip']
assert len(RUN_ROOTS) + len(RUN_ARCHIVES) == 1, (
    'Attach exactly one completed Food output, either as an extracted artifact dataset or as '
    'green_sm_food_poc20_v1_r1.zip. '
    f'Found roots={RUN_ROOTS}, archives={RUN_ARCHIVES}'
)
RUN_ROOT = RUN_ROOTS[0] if RUN_ROOTS else None
RUN_ARCHIVE = RUN_ARCHIVES[0] if RUN_ARCHIVES else None
print('Food PoC:', POC_ROOT)
print('Completed Food output:', RUN_ROOT or RUN_ARCHIVE)


In [ ]:
if WORK_ROOT.exists():
    raise RuntimeError(f'Fresh Kaggle session required: {WORK_ROOT} already exists.')
WORK_ROOT.mkdir(parents=True)

if RUN_ROOT is not None:
    RUN_DIR = RUN_ROOT
    print('Using mounted completed Food run:', RUN_DIR)
else:
    assert RUN_ARCHIVE is not None
    RUN_DIR = WORK_ROOT / 'completed_food_run'
    RUN_DIR.mkdir()
    with zipfile.ZipFile(RUN_ARCHIVE) as archive:
        for member in archive.infolist():
            relative = PurePosixPath(member.filename)
            assert not relative.is_absolute() and '..' not in relative.parts, f'Unsafe ZIP entry: {member.filename}'
        archive.extractall(RUN_DIR)
    print('Extracted completed Food run:', RUN_DIR)

for relative in (
    'run_manifest.json',
    'predictions/test/predictions.test.parquet',
    'predictions/test/run_manifest.test.json',
    'predictions/test/validation_report.test.json',
):
    assert (RUN_DIR / relative).is_file(), f'Missing completed Food artifact: {relative}'
print('Verified completed Food run:', RUN_DIR)


In [ ]:
import json

REPORT_PATH = WORK_ROOT / 'food_test_evaluation.json'
subprocess.run([
    sys.executable, '-m', 'src.evaluation', 'evaluate',
    '--task', 'food_recommendation',
    '--split', 'test',
    '--prediction-dir', str(RUN_DIR / 'predictions' / 'test'),
    '--opportunities-root', str(POC_ROOT / 'food_opportunities'),
    '--output', str(REPORT_PATH),
], cwd=REPO, check=True)

report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
print(json.dumps(report, indent=2, sort_keys=True))


In [ ]:
import shutil

archive = shutil.make_archive('/kaggle/working/green_sm_food_poc20_evaluation', 'zip', root_dir=WORK_ROOT)
print('Download evaluation report:', archive)
